In [2]:
from kafka import KafkaProducer
import json
import time
import random
from math import cos, pi

In [3]:
def generate_sensor_data():
    timestamp = int(time.time())

    # Simulate IoT sensor data for water quality metrics with realistic patterns
    water_temperature = random.uniform(1, 3) + round(20 + 5 * (1 + 0.5 * (1 + cos((timestamp % 86400) / 86400 * 2 * pi))), 2)
    ph_level = random.uniform(0, 1) + round(7.5 + 0.2 * (1 + cos((timestamp % 86400) / 86400 * 2 * pi)), 2)
    turbidity = round(random.uniform(5, 50), 2)  # Turbidity in NTU (Nephelometric Turbidity Units)
    dissolved_oxygen = round(random.uniform(5, 12), 2)  # Dissolved Oxygen in mg/L

    return {
        "timestamp": timestamp,
        "water_temperature": water_temperature,
        "ph_level": ph_level,
        "turbidity": turbidity,
        "dissolved_oxygen": dissolved_oxygen
    }

In [4]:
import numpy as np
import time

def generate_sensor_data_np(n_samples=100): # n_samples define el tamaño del array
    timestamps = np.arange(int(time.time()), int(time.time()) + n_samples)
    
    phase = (timestamps % 86400) / 86400 * 2 * np.pi
    cos_phase = np.cos(phase)

    water_temp = np.random.uniform(1, 3, n_samples) + np.round(20 + 5 * (1 + 0.5 * (1 + cos_phase)), 2)
    ph_level = np.random.uniform(0, 1, n_samples) + np.round(7.5 + 0.2 * (1 + cos_phase), 2)
    turbidity = np.round(np.random.uniform(5, 50, n_samples), 2)
    dissolved_oxygen = np.round(np.random.uniform(5, 12, n_samples), 2)

    data = np.rec.fromarrays(
        [timestamps, water_temp, ph_level, turbidity, dissolved_oxygen],
        names='timestamp, water_temperature, ph_level, turbidity, dissolved_oxygen'
    )
    return data

sensor_data = generate_sensor_data_np(50)
print(sensor_data['water_temperature'])

[27.89164434 28.66510943 28.47488558 29.5600414  28.60174118 28.16505753
 28.03334795 28.39310904 28.39450896 27.66144568 28.18752387 29.27774491
 28.0785007  27.90577821 28.08634335 27.97933058 29.40403817 28.79804287
 29.3365096  28.15058707 27.93075805 29.01344933 28.04363909 29.3101132
 29.62318112 29.60162051 28.20128242 28.28011014 29.32368512 28.38196857
 27.84245638 28.51407732 29.49096951 29.04452198 29.25262176 28.79636574
 29.63907685 27.69986995 29.21982609 28.57078839 27.99439437 28.8863383
 29.38713375 27.83075833 29.00022008 28.32120772 29.19058694 27.95493001
 28.9962913  29.50420136]


In [14]:
 # Kafka configuration
kafka_topic = "water_quality"
kafka_bootstrap_servers = ["localhost:9092"] 

# Create Kafka producer
producer = KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print(f"Producing messages to Kafka topic '{kafka_topic}'...")

try:
    while True:
        # Generate sensor data
        sensor_data = generate_sensor_data()

        # Publish sensor data to Kafka
        producer.send(kafka_topic, sensor_data)

        print(f"Sent: {sensor_data}")

        # Wait for 1 second
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopped producing messages.")
finally:
    producer.close()

Producing messages to Kafka topic 'water_quality'...
Sent: {'timestamp': 1772332568, 'water_temperature': 31.327524809242767, 'ph_level': 8.328555078754544, 'turbidity': 15.01, 'dissolved_oxygen': 6.79}
Sent: {'timestamp': 1772332569, 'water_temperature': 31.80051913937862, 'ph_level': 8.355714388032414, 'turbidity': 13.97, 'dissolved_oxygen': 5.49}
Sent: {'timestamp': 1772332570, 'water_temperature': 30.990254511052346, 'ph_level': 8.325039702839, 'turbidity': 6.68, 'dissolved_oxygen': 10.59}
Sent: {'timestamp': 1772332571, 'water_temperature': 31.18845228390399, 'ph_level': 8.494123940306201, 'turbidity': 42.8, 'dissolved_oxygen': 10.96}
Sent: {'timestamp': 1772332572, 'water_temperature': 31.81327952504047, 'ph_level': 8.4028558849549, 'turbidity': 48.9, 'dissolved_oxygen': 6.15}
Sent: {'timestamp': 1772332573, 'water_temperature': 30.772680780021094, 'ph_level': 8.495080619207096, 'turbidity': 9.64, 'dissolved_oxygen': 5.15}
Stopped producing messages.


In [15]:
 # Kafka configuration
kafka_topic = "water_quality"
kafka_bootstrap_servers = ["localhost:9092"] 

# Create Kafka producer
producer = KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print(f"Producing messages to Kafka topic '{kafka_topic}'...")

try:
    while True:
        # Generate sensor data
        sensor_data = generate_sensor_data_np()

        # Publish sensor data to Kafka
        producer.send(kafka_topic, sensor_data)

        print(f"Sent: {sensor_data}")

        # Wait for 1 second
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopped producing messages.")
finally:
    producer.close()

Producing messages to Kafka topic 'water_quality'...


TypeError: Object of type recarray is not JSON serializable

In [ ]:
 # Kafka configuration
import sys
kafka_topic = "water_quality"
kafka_bootstrap_servers = ["localhost:9092"] 
producer_number = 4
producers = []
producer_ids = []
for i in range(producer_number):
    print(f"Creating producer {i} for '{kafka_topic} topic'...")
    producer_ids.append(str(i))
    producers.append(KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
))

try:
    while True:
        for i in range(producer_number):
            # Generate sensor data
            raw_data = generate_sensor_data()
            sensor_data = {
            'sensor_id': producer_ids[i],
            **raw_data
            }
            # Publish sensor data to Kafka
            print(f'Publishing sensor data on sensor {producer_ids[i]}')
            producers[i].send(kafka_topic,
            key=producer_ids[i].encode('utf-8'),value=sensor_data)
            print(f"Sent: {sensor_data}")
            time.sleep(1)
except KeyboardInterrupt:
    print("Stopped producing messages.")
finally:
    producer.close()

Creating producer 0 for 'water_quality topic'...
Creating producer 1 for 'water_quality topic'...
Creating producer 2 for 'water_quality topic'...
Creating producer 3 for 'water_quality topic'...
Publishing sensor data on sensor 0
Sent: {'sensor_id': '0', 'timestamp': 1772383296, 'water_temperature': 28.014960632761472, 'ph_level': 7.860216409478289, 'turbidity': 48.98, 'dissolved_oxygen': 11.26}
Publishing sensor data on sensor 1
Sent: {'sensor_id': '1', 'timestamp': 1772383297, 'water_temperature': 27.763691235845663, 'ph_level': 8.266927889050859, 'turbidity': 7.36, 'dissolved_oxygen': 7.9}
Publishing sensor data on sensor 2
Sent: {'sensor_id': '2', 'timestamp': 1772383298, 'water_temperature': 27.751688278057, 'ph_level': 7.857403303049662, 'turbidity': 37.36, 'dissolved_oxygen': 7.55}
Publishing sensor data on sensor 3
Sent: {'sensor_id': '3', 'timestamp': 1772383299, 'water_temperature': 28.338931516774228, 'ph_level': 8.232513666854379, 'turbidity': 6.33, 'dissolved_oxygen': 11.